# Notebook 02 — Detector por Reglas Simples
Detecta anomalías en `data/metadata.csv` usando 3 reglas basadas en umbrales.

In [1]:
import pandas as pd

df = pd.read_csv('../data/metadata.csv')
print(f'Filas cargadas: {len(df)}')
df.head()

Filas cargadas: 50


,mesa_id,municipio,candidato_1,candidato_2,candidato_3,candidato_4,candidato_5,votos_nulos,total_votos,capacidad_mesa,tiene_anomalia,tipo_anomalia
0,Mesa_001,San Marcos,24,13,45,41,38,4,165,200,False,NaN
1,Mesa_002,La Esperanza,23,79,21,64,14,0,201,200,False,NaN
2,Mesa_003,Villa Nueva,49,36,40,26,19,0,170,200,True,votos_nulos_cero
3,Mesa_004,El Progreso,35,79,63,38,67,18,300,200,False,NaN
4,Mesa_005,Santa Rosa,45,10,30,64,53,8,210,200,False,NaN


In [2]:
# Regla 1: suma real de candidatos + votos_nulos != total_votos del CSV
cols_candidatos = ['candidato_1','candidato_2','candidato_3','candidato_4','candidato_5']
df['suma_real'] = df[cols_candidatos].sum(axis=1) + df['votos_nulos']
df['alerta_total'] = df['suma_real'] != df['total_votos']

# Regla 2: tasa de participación > 0.95
df['tasa_participacion'] = df['total_votos'] / df['capacidad_mesa']
df['alerta_participacion'] = df['tasa_participacion'] > 0.95

# Regla 3: votos_nulos == 0 y total_votos > 150
df['alerta_nulos'] = (df['votos_nulos'] == 0) & (df['total_votos'] > 150)

# Bandera general
df['sospechosa_reglas'] = df['alerta_total'] | df['alerta_participacion'] | df['alerta_nulos']

In [3]:
print('=== Resumen por regla ===')
resumen_reglas = pd.DataFrame({
    'Regla': ['alerta_total', 'alerta_participacion', 'alerta_nulos', 'sospechosa_reglas'],
    'Mesas marcadas': [
        df['alerta_total'].sum(),
        df['alerta_participacion'].sum(),
        df['alerta_nulos'].sum(),
        df['sospechosa_reglas'].sum(),
    ]
})
print(resumen_reglas.to_string(index=False))

=== Resumen por regla ===
               Regla  Mesas marcadas
        alerta_total               5
alerta_participacion              39
        alerta_nulos               5
   sospechosa_reglas              42


In [4]:
print('=== Métricas de detección ===')
vp = ((df['sospechosa_reglas'] == True)  & (df['tiene_anomalia'] == True)).sum()
fp = ((df['sospechosa_reglas'] == True)  & (df['tiene_anomalia'] == False)).sum()
fn = ((df['sospechosa_reglas'] == False) & (df['tiene_anomalia'] == True)).sum()
vn = ((df['sospechosa_reglas'] == False) & (df['tiene_anomalia'] == False)).sum()

print(f'Verdaderos Positivos (VP): {vp}')
print(f'Falsos Positivos    (FP): {fp}')
print(f'Falsos Negativos    (FN): {fn}')
print(f'Verdaderos Negativos(VN): {vn}')
precision = vp / (vp + fp) if (vp + fp) > 0 else 0
recall    = vp / (vp + fn) if (vp + fn) > 0 else 0
print(f'Precisión: {precision:.2f} | Recall: {recall:.2f}')

=== Métricas de detección ===
Verdaderos Positivos (VP): 10
Falsos Positivos    (FP): 32
Falsos Negativos    (FN): 0
Verdaderos Negativos(VN): 8
Precisión: 0.24 | Recall: 1.00


In [5]:
df.to_csv('../data/resultados_reglas.csv', index=False)
print('Exportado: ../data/resultados_reglas.csv')
print(f'Columnas: {list(df.columns)}')

Exportado: ../data/resultados_reglas.csv
Columnas: ['mesa_id', 'municipio', 'candidato_1', 'candidato_2', 'candidato_3', 'candidato_4', 'candidato_5', 'votos_nulos', 'total_votos', 'capacidad_mesa', 'tiene_anomalia', 'tipo_anomalia', 'suma_real', 'alerta_total', 'tasa_participacion', 'alerta_participacion', 'alerta_nulos', 'sospechosa_reglas']
